In [1]:
import os
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (9, 4.5), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.titlesize": 12, "axes.titleweight": "bold", "axes.labelsize": 10})
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# ---- KONFIGURASI (ubah di sini) ----
TOP_N = 6                  # jumlah komoditas teratas pada grafik tren
TAHUN_GEO = None           # tahun untuk analisis geografis; None = tahun terakhir
Z_AMBANG = 3.5             # ambang robust z-score (median/MAD) untuk anomali produktivitas
LONJAKAN_YOY = 0.5         # perubahan produksi antar tahun > 50% dianggap lonjakan
LUAS_KECIL_HA = 0.5        # luas di bawah ini dianggap sangat kecil
SERTAKAN_JAMUR_M2 = False  # False = baris berluas m² (jamur) tidak ikut, karena satuannya beda dengan hektar

# ---- LOKASI PROJECT ----
search_dir, PROJECT_ROOT = os.getcwd(), None
while True:
    if os.path.isdir(os.path.join(search_dir, "data")):
        PROJECT_ROOT = search_dir
        break
    if os.path.dirname(search_dir) == search_dir:
        break
    search_dir = os.path.dirname(search_dir)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Folder 'data' tidak ditemukan. Jalankan notebook dari dalam repository.")
DATA_DIR = os.path.join(PROJECT_ROOT, "data")

# ---- CARI FILE PRODUKTIVITAS (buah & sayur, per kecamatan & kabupaten) ----
terpilih = {}
for root, _, files in os.walk(DATA_DIR):
    for f in files:
        n = f.lower()
        if n.endswith(".csv") and "produktivitas" in n and ("buah" in n or "sayur" in n):
            kelompok = "Buah" if "buah" in n else "Sayur"
            level = "kecamatan" if "kec" in n else "kabupaten"
            p = os.path.join(root, f)
            kunci = (kelompok, level)
            if kunci not in terpilih or os.path.getmtime(p) > os.path.getmtime(terpilih[kunci]):
                terpilih[kunci] = p            # kalau ada salinan ganda, pakai yang terbaru
if not any(l == "kecamatan" for _, l in terpilih):
    raise FileNotFoundError("File produktivitas per kecamatan (buah/sayur) tidak ditemukan di data/. "
                            "Jalankan dulu notebook perhitungan produktivitas.")

ALIAS = {
    "tahun": ["tahun"], "kecamatan": ["kecamatan"], "komoditas": ["komoditas", "tanaman"],
    "luas_panen": ["luas_panen_ha", "luas_panen"], "produksi_kuintal": ["produksi_kuintal", "produksi"],
    "produktivitas": ["produktivitas_kuintal_ha", "produktivitas"], "satuan_luas": ["satuan_luas", "satuan"],
    "flag_kualitas": ["flag_kualitas"], "is_dummy": ["is_dummy"],
}


def muat(path, kelompok, level):
    raw = pd.read_csv(path, encoding="utf-8-sig")
    raw.columns = [re.sub(r"\W+", "_", str(c).strip().lower()).strip("_") for c in raw.columns]
    baku, dipakai = {}, set()
    for k, alias in ALIAS.items():
        ada = [a for a in alias if a in raw.columns]
        if ada:
            baku[k] = raw[ada[0]]
            dipakai.update(ada)
    wajib = ["tahun", "komoditas", "luas_panen", "produksi_kuintal"] + (["kecamatan"] if level == "kecamatan" else [])
    hilang = [w for w in wajib if w not in baku]
    if hilang:
        raise ValueError(f"{os.path.basename(path)}: kolom wajib tidak ditemukan: {hilang}. "
                         f"Kolom yang ada: {list(raw.columns)}")
    df = pd.concat([pd.DataFrame(baku), raw[[c for c in raw.columns if c not in dipakai]]], axis=1)
    df["tahun"] = pd.to_numeric(df["tahun"]).astype(int)
    for c in ["luas_panen", "produksi_kuintal", "produktivitas"]:
        if c in df:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    if "satuan_luas" not in df:
        df["satuan_luas"] = "Ha"
    df["satuan_luas"] = df["satuan_luas"].astype(str).str.strip().str.title().replace({"M2": "M2", "Ha": "Ha"})
    if "produktivitas" not in df:
        df["produktivitas"] = np.where((df["luas_panen"] > 0) & (df["produksi_kuintal"] > 0),
                                       df["produksi_kuintal"] / df["luas_panen"], np.nan)
    if "flag_kualitas" not in df:
        df["flag_kualitas"] = "tidak_tersedia"
    df["is_dummy"] = df["is_dummy"].astype(str).str.lower().isin(["true", "1"]) if "is_dummy" in df else False
    df["kelompok"] = kelompok
    return df


DF_KEC = pd.concat([muat(p, k, l) for (k, l), p in terpilih.items() if l == "kecamatan"], ignore_index=True)
kab = [muat(p, k, l) for (k, l), p in terpilih.items() if l == "kabupaten"]
DF_KAB = pd.concat(kab, ignore_index=True) if kab else pd.DataFrame()

for (k, l), p in sorted(terpilih.items()):
    print(f"✓ {k:<5} {l:<9} <- {os.path.relpath(p, PROJECT_ROOT)}")

# ---- DATASET ANALISIS ----
DF = DF_KEC.copy()
if not SERTAKAN_JAMUR_M2:
    n_m2 = (DF["satuan_luas"] != "Ha").sum()
    DF = DF[DF["satuan_luas"] == "Ha"].copy()
    if n_m2:
        print(f"ℹ️  {n_m2} baris berluas m² (jamur) dikeluarkan dari EDA karena satuannya beda dengan hektar.")

DF_AKTIF = DF[(DF["luas_panen"] > 0) & (DF["produksi_kuintal"] > 0)].copy()   # baris yang benar-benar ada panen
TAHUN_DUMMY = sorted(DF.loc[DF["is_dummy"], "tahun"].unique().tolist())

print(f"\nBaris total       : {len(DF):,}  (kecamatan × komoditas × tahun)")
print(f"Baris ada panen   : {len(DF_AKTIF):,}  ({len(DF_AKTIF) / len(DF):.1%}); sisanya luas = 0 dan produksi = 0")
print(f"Tahun             : {DF['tahun'].min()} - {DF['tahun'].max()}  ({DF['tahun'].nunique()} tahun)")
print(f"Kecamatan         : {DF['kecamatan'].nunique()} | Komoditas: {DF['komoditas'].nunique()} | Kelompok: {sorted(DF['kelompok'].unique())}")
if TAHUN_DUMMY:
    print(f"\n⚠️  Tahun {TAHUN_DUMMY} adalah DATA SIMULASI (is_dummy=True). Tren dan pola antar tahun "
          "hanya ilustrasi metode, jangan dipresentasikan sebagai kondisi nyata.")

ValueError: Produktivitas_Hortikultura_Buah_per_kecamatan.csv: kolom wajib tidak ditemukan: ['luas_panen', 'produksi_kuintal']. Kolom yang ada: ['tahun', 'kecamatan', 'komoditas', 'produktivitas_kuintal_ha']

In [ ]:
VAR = {"luas_panen": "Luas panen (ha)", "produksi_kuintal": "Produksi (kuintal)",
       "produktivitas": "Produktivitas (kuintal/ha)"}


def interpretasi(teks):
    print("📝 Interpretasi:", teks, "\n")


def tandai_dummy(ax):
    if TAHUN_DUMMY:
        ax.axvspan(min(TAHUN_DUMMY) - 0.5, max(TAHUN_DUMMY) + 0.5, color="orange", alpha=0.12,
                   label="Tahun data simulasi")


def batas_iqr(x, k=1.5):
    q1, q3 = x.quantile([0.25, 0.75])
    return q1 - k * (q3 - q1), q3 + k * (q3 - q1)


def robust_z(x):
    med = x.median()
    mad = (x - med).abs().median()
    return 0.6745 * (x - med) / mad if mad > 0 else pd.Series(0.0, index=x.index)


def heatmap(ax, mat, judul, label_cbar, cmap="RdBu_r", vmin=None, vmax=None, anotasi_min=None, fmt="{:.2f}"):
    im = ax.imshow(mat.values, cmap=cmap, vmin=vmin, vmax=vmax, aspect="auto")
    ax.set_xticks(range(mat.shape[1]))
    ax.set_xticklabels(mat.columns, rotation=35, ha="right")
    ax.set_yticks(range(mat.shape[0]))
    ax.set_yticklabels(mat.index)
    ax.grid(False)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            v = mat.values[i, j]
            if np.isfinite(v) and (anotasi_min is None or v >= anotasi_min):
                ax.text(j, i, fmt.format(v), ha="center", va="center", fontsize=7)
    ax.set_title(judul)
    plt.colorbar(im, ax=ax, label=label_cbar)


def label_tahun(tahun):
    return [f"{t}*" if t in TAHUN_DUMMY else str(t) for t in tahun]


print("✓ Fungsi bantu siap.")

In [ ]:
# 3.1 Statistik deskriptif (hanya baris yang ada panen)
ringkas = DF_AKTIF[list(VAR)].describe(percentiles=[0.25, 0.5, 0.75, 0.95]).T
ringkas["skewness"] = DF_AKTIF[list(VAR)].skew()
ringkas["cv"] = ringkas["std"] / ringkas["mean"]
print("Distribusi variabel numerik (baris dengan panen):")
display(ringkas)
interpretasi("Selisih besar antara rata-rata dan median serta skewness positif menunjukkan sebaran sangat miring ke kanan: "
             "sebagian besar kombinasi kecamatan-komoditas bernilai kecil, sedikit yang sangat besar. "
             "Karena itu histogram dan boxplot di bawah memakai skala logaritma.")

# 3.2 Histogram (sumbu x skala log)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (kolom, label) in zip(axes, VAR.items()):
    x = DF_AKTIF[kolom].dropna()
    ax.hist(x, bins=np.logspace(np.log10(x.min()), np.log10(x.max()), 30), color="steelblue", edgecolor="white")
    ax.axvline(x.median(), color="red", ls="--", label=f"Median = {x.median():,.1f}")
    ax.set_xscale("log")
    ax.set_title(f"Distribusi {label}")
    ax.set_xlabel(f"{label} (skala log)")
    ax.set_ylabel("Jumlah baris")
    ax.legend()
plt.tight_layout()
plt.show()
for kolom, label in VAR.items():
    x = DF_AKTIF[kolom].dropna()
    interpretasi(f"{label}: median {x.median():,.1f}, rata-rata {x.mean():,.1f}, persentil 95 = {x.quantile(.95):,.1f}, "
                 f"maksimum {x.max():,.1f} (skewness {x.skew():.1f}).")

# 3.3 Boxplot per kelompok (sumbu y skala log)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
kelompok_list = sorted(DF_AKTIF["kelompok"].unique())
for ax, (kolom, label) in zip(axes, VAR.items()):
    ax.boxplot([DF_AKTIF.loc[DF_AKTIF["kelompok"] == k, kolom].dropna() for k in kelompok_list])
    ax.set_xticklabels(kelompok_list)
    ax.set_yscale("log")
    ax.set_title(f"Boxplot {label}")
    ax.set_xlabel("Kelompok hortikultura")
    ax.set_ylabel(f"{label} (skala log)")
plt.tight_layout()
plt.show()
med = DF_AKTIF.groupby("kelompok")[list(VAR)].median().round(1)
display(med)
interpretasi("Titik di luar kotak adalah pencilan menurut aturan 1,5×IQR pada tiap kelompok. Bandingkan median antar kelompok "
             "pada tabel di atas untuk melihat kelompok mana yang cenderung lebih luas, lebih produktif, atau lebih besar produksinya.")

# 3.4 Frekuensi variabel kategorikal
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
DF["flag_kualitas"].value_counts().sort_values().plot.barh(ax=axes[0, 0], color="slateblue")
axes[0, 0].set_title("Frekuensi flag kualitas data")
axes[0, 0].set_xlabel("Jumlah baris"); axes[0, 0].set_ylabel("Flag kualitas")

DF_AKTIF["komoditas"].value_counts().head(15).sort_values().plot.barh(ax=axes[0, 1], color="seagreen")
axes[0, 1].set_title("15 komoditas dengan catatan panen terbanyak")
axes[0, 1].set_xlabel("Jumlah baris berpanen (kecamatan × tahun)"); axes[0, 1].set_ylabel("Komoditas")

DF.groupby("kelompok")["komoditas"].nunique().plot.bar(ax=axes[1, 0], color="darkorange", rot=0)
axes[1, 0].set_title("Jumlah komoditas per kelompok")
axes[1, 0].set_xlabel("Kelompok"); axes[1, 0].set_ylabel("Jumlah komoditas")

DF_AKTIF.groupby("tahun").size().plot.bar(ax=axes[1, 1], color="gray", rot=0)
axes[1, 1].set_title("Jumlah baris berpanen per tahun")
axes[1, 1].set_xlabel("Tahun"); axes[1, 1].set_ylabel("Jumlah baris")
plt.tight_layout()
plt.show()
interpretasi(f"{1 - len(DF_AKTIF) / len(DF):.0%} kombinasi kecamatan-komoditas-tahun tidak berpanen (luas dan produksi 0), "
             "artinya banyak komoditas hanya ditanam di sebagian kecil kecamatan. Komoditas dengan catatan terbanyak adalah yang paling tersebar.")

# 3.5 Identifikasi outlier (IQR 1,5 pada skala log)
baris = []
for kolom, label in VAR.items():
    x = np.log10(DF_AKTIF[kolom].dropna())
    bawah, atas = batas_iqr(x)
    n_out = ((x < bawah) | (x > atas)).sum()
    baris.append({"variabel": label, "batas_bawah": 10 ** bawah, "batas_atas": 10 ** atas,
                  "jumlah_outlier": n_out, "persen_outlier": n_out / len(x) * 100})
display(pd.DataFrame(baris))
kolom_tampil = ["tahun", "kecamatan", "komoditas", "luas_panen", "produksi_kuintal", "produktivitas"]
print("10 produksi terbesar:")
display(DF_AKTIF.nlargest(10, "produksi_kuintal")[kolom_tampil])
interpretasi("Outlier di sini dihitung terhadap seluruh komoditas sekaligus, sehingga komoditas bervolume besar (mis. yang sentra di satu kecamatan) "
             "otomatis tampak sebagai outlier. Penilaian anomali yang adil per komoditas ada di bagian 5.")

In [ ]:
# 4.1 Korelasi antarvariabel numerik + heatmap
nama_pendek = {"luas_panen": "Luas panen", "produksi_kuintal": "Produksi", "produktivitas": "Produktivitas"}
log_data = np.log10(DF_AKTIF[list(VAR)]).rename(columns=nama_pendek)
pearson = log_data.corr(method="pearson")
spearman = DF_AKTIF[list(VAR)].rename(columns=nama_pendek).corr(method="spearman")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
heatmap(axes[0], pearson, "Korelasi Pearson (data log10)", "Koefisien korelasi", vmin=-1, vmax=1)
heatmap(axes[1], spearman, "Korelasi Spearman (data asli)", "Koefisien korelasi", vmin=-1, vmax=1)
plt.tight_layout()
plt.show()
pasangan = pearson.where(np.triu(np.ones(pearson.shape), 1).astype(bool)).stack().sort_values(key=abs, ascending=False)
(a, b), r = pasangan.index[0], pasangan.iloc[0]
interpretasi(f"Hubungan terkuat ada antara {a.lower()} dan {b.lower()} (Pearson log = {r:.2f}). "
             f"Korelasi luas panen–produksi {pearson.loc['Luas panen', 'Produksi']:.2f}, luas panen–produktivitas "
             f"{pearson.loc['Luas panen', 'Produktivitas']:.2f}. Korelasi bukan bukti sebab-akibat.")

# 4.2 Hubungan luas panen dan produksi (log-log) + garis regresi
fig, ax = plt.subplots(figsize=(8, 5.5))
warna = {"Buah": "tab:green", "Sayur": "tab:blue"}
kalimat = []
for k, g in DF_AKTIF.groupby("kelompok"):
    ax.scatter(g["luas_panen"], g["produksi_kuintal"], s=8, alpha=0.35, color=warna.get(k), label=f"{k} (n={len(g):,})")
    lx, ly = np.log10(g["luas_panen"]), np.log10(g["produksi_kuintal"])
    slope, intercept = np.polyfit(lx, ly, 1)
    xs = np.linspace(lx.min(), lx.max(), 50)
    ax.plot(10 ** xs, 10 ** (slope * xs + intercept), color=warna.get(k), lw=2, label=f"Regresi {k} (kemiringan {slope:.2f})")
    kalimat.append(f"{k}: kemiringan {slope:.2f}, r = {np.corrcoef(lx, ly)[0, 1]:.2f}")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_title("Hubungan luas panen dan produksi")
ax.set_xlabel("Luas panen (ha, skala log)"); ax.set_ylabel("Produksi (kuintal, skala log)")
ax.legend()
plt.show()
interpretasi("Garis regresi pada skala log: kemiringan mendekati 1 berarti produksi naik hampir proporsional dengan luas panen "
             "(produktivitas relatif seragam). " + "; ".join(kalimat) + ".")

# 4.3 Hubungan variabel cuaca dan produksi (hanya jika datanya ada di dataset)
kolom_cuaca = [c for c in DF.columns if re.search(r"suhu|curah|hujan|kelembaban|angin|cuaca", c)]
if kolom_cuaca:
    fig, axes = plt.subplots(1, min(len(kolom_cuaca), 3), figsize=(5 * min(len(kolom_cuaca), 3), 4), squeeze=False)
    for ax, c in zip(axes[0], kolom_cuaca[:3]):
        d = DF_AKTIF[[c, "produksi_kuintal"]].dropna()
        ax.scatter(d[c], d["produksi_kuintal"], s=8, alpha=0.4)
        ax.set_yscale("log")
        ax.set_title(f"{c} vs produksi"); ax.set_xlabel(c); ax.set_ylabel("Produksi (kuintal, skala log)")
        interpretasi(f"Korelasi Spearman {c} – produksi = {d[c].corr(d['produksi_kuintal'], method='spearman'):.2f}.")
    plt.tight_layout(); plt.show()
else:
    print("ℹ️  Variabel cuaca (suhu, curah hujan, dst.) tidak ada pada dataset ini, bagian ini dilewati. "
          "Tambahkan kolomnya (per kecamatan-tahun) agar analisis ini aktif otomatis.\n")

# 4.3b Hubungan waktu dan harga pangan (hanya jika datanya ada)
kolom_harga = [c for c in DF.columns if "harga" in c]
if kolom_harga:
    fig, ax = plt.subplots()
    for c in kolom_harga[:3]:
        DF.groupby("tahun")[c].mean().plot(ax=ax, marker="o", label=c)
    ax.set_title("Tren harga rata-rata per tahun"); ax.set_xlabel("Tahun"); ax.set_ylabel("Harga rata-rata"); ax.legend()
    plt.show()
    interpretasi("Tren harga rata-rata per tahun dari kolom harga pada dataset.")
else:
    print("ℹ️  Data harga pangan tidak ada pada dataset ini, bagian ini dilewati.\n")

In [ ]:
tahun_urut = sorted(DF["tahun"].unique())
if len(tahun_urut) < 2:
    print("ℹ️  Hanya ada satu tahun pada dataset, analisis temporal dilewati.")
else:
    tp = DF.groupby(["tahun", "kelompok"])[["luas_panen", "produksi_kuintal"]].sum().reset_index()

    def tren_total(kolom, judul, label_y, skala):
        fig, ax = plt.subplots()
        for k, g in tp.groupby("kelompok"):
            ax.plot(g["tahun"], g[kolom] / skala, marker="o", label=k)
        tandai_dummy(ax)
        ax.set_title(judul); ax.set_xlabel("Tahun"); ax.set_ylabel(label_y)
        ax.set_xticks(tahun_urut); ax.legend()
        plt.show()
        kal = []
        for k, g in tp.groupby("kelompok"):
            g = g.sort_values("tahun")
            awal, akhir = g[kolom].iloc[0], g[kolom].iloc[-1]
            kal.append(f"{k}: {awal / skala:,.1f} → {akhir / skala:,.1f} ({(akhir / awal - 1) * 100:+.1f}%) dari {g['tahun'].iloc[0]} ke {g['tahun'].iloc[-1]}, "
                       f"puncak di {int(g.loc[g[kolom].idxmax(), 'tahun'])}")
        return "; ".join(kal)

    # 5.1 Tren produksi, 5.2 Tren luas panen
    kal = tren_total("produksi_kuintal", "Tren total produksi hortikultura per tahun", "Produksi (ribu kuintal)", 1e3)
    interpretasi(kal + ". Area oranye = tahun data simulasi." if TAHUN_DUMMY else kal + ".")
    kal = tren_total("luas_panen", "Tren total luas panen hortikultura per tahun", "Luas panen (ribu ha)", 1e3)
    interpretasi(kal + ". Area oranye = tahun data simulasi." if TAHUN_DUMMY else kal + ".")

    # 5.3 Tren per komoditas teratas
    tahun_akhir = tahun_urut[-1]
    top = (DF[DF["tahun"] == tahun_akhir].groupby("komoditas")["produksi_kuintal"].sum().nlargest(TOP_N).index)
    pk = DF[DF["komoditas"].isin(top)].groupby(["tahun", "komoditas"])["produksi_kuintal"].sum().unstack()
    fig, ax = plt.subplots(figsize=(9, 5))
    for k in top:
        ax.plot(pk.index, pk[k] / 1e3, marker="o", label=k)
    tandai_dummy(ax)
    ax.set_yscale("log")
    ax.set_title(f"Tren produksi {TOP_N} komoditas terbesar (menurut {tahun_akhir})")
    ax.set_xlabel("Tahun"); ax.set_ylabel("Produksi (ribu kuintal, skala log)")
    ax.set_xticks(tahun_urut); ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left")
    plt.show()
    interpretasi(f"Komoditas terbesar pada {tahun_akhir} adalah {top[0]}. Garis yang hampir mendatar berarti produksi stabil antar tahun; "
                 "garis yang naik-turun tajam perlu dicek apakah mencerminkan kondisi nyata atau variasi data simulasi.")

    # 5.4 Fluktuasi: perubahan antar tahun & volatilitas
    pivot = tp.pivot(index="tahun", columns="kelompok", values="produksi_kuintal")
    yoy = pivot.pct_change().dropna() * 100
    fig, ax = plt.subplots()
    x = np.arange(len(yoy))
    lebar = 0.8 / yoy.shape[1]
    for i, k in enumerate(yoy.columns):
        ax.bar(x + i * lebar, yoy[k], lebar, label=k)
    ax.axhline(0, color="black", lw=0.8)
    ax.set_xticks(x + lebar * (yoy.shape[1] - 1) / 2); ax.set_xticklabels(label_tahun(yoy.index))
    ax.set_title("Perubahan total produksi dibanding tahun sebelumnya")
    ax.set_xlabel("Tahun (* = data simulasi)"); ax.set_ylabel("Perubahan (%)"); ax.legend()
    plt.show()
    interpretasi(f"Perubahan tahunan terbesar: {yoy.abs().stack().idxmax()[1]} pada {yoy.abs().stack().idxmax()[0]} "
                 f"({yoy.stack()[yoy.abs().stack().idxmax()]:+.1f}%).")

    per_komoditas = DF.groupby(["komoditas", "tahun"])["produksi_kuintal"].sum().unstack()
    per_komoditas = per_komoditas[(per_komoditas > 0).sum(axis=1) >= 3]
    cv = (per_komoditas.std(axis=1) / per_komoditas.mean(axis=1)).sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(8, 5))
    cv.head(10).sort_values().plot.barh(ax=ax, color="indianred")
    ax.set_title("10 komoditas dengan produksi paling berfluktuasi antar tahun")
    ax.set_xlabel("Koefisien variasi (simpangan baku ÷ rata-rata)"); ax.set_ylabel("Komoditas")
    plt.show()
    interpretasi(f"Komoditas paling berfluktuasi: {cv.index[0]} (CV {cv.iloc[0]:.2f}); paling stabil: {cv.index[-1]} (CV {cv.iloc[-1]:.2f}). "
                 "Komoditas dengan produksi sedikit dan tidak ada di semua tahun cenderung tampak fluktuatif.")

    # 5.5 Tren harga (hanya jika ada)
    if not [c for c in DF.columns if "harga" in c]:
        print("ℹ️  Data harga komoditas tidak ada pada dataset ini, tren harga dilewati.\n")

In [ ]:
tahun_geo = int(TAHUN_GEO) if TAHUN_GEO else int(DF["tahun"].max())
G = DF[DF["tahun"] == tahun_geo]
print(f"Tahun yang dianalisis: {tahun_geo}" + ("  ⚠️ data simulasi" if tahun_geo in TAHUN_DUMMY else ""))

if len(DF_KAB):
    print("\nTingkat kabupaten (satu wilayah: Kabupaten Kediri), total per kelompok:")
    kab_ringkas = DF_KAB[DF_KAB["satuan_luas"] == "Ha"].groupby(["tahun", "kelompok"])[["luas_panen", "produksi_kuintal"]].sum()
    display(kab_ringkas.loc[tahun_geo] if tahun_geo in kab_ringkas.index.get_level_values(0) else kab_ringkas.tail(4))
print("Perbandingan wilayah di bawah dilakukan antar kecamatan.\n")

# 6.1 Produksi per kecamatan (ditumpuk per kelompok)
prod = G.pivot_table(index="kecamatan", columns="kelompok", values="produksi_kuintal", aggfunc="sum", fill_value=0)
prod["Total"] = prod.sum(axis=1)
prod = prod.sort_values("Total")
fig, ax = plt.subplots(figsize=(9, 8))
prod.drop(columns="Total").plot.barh(stacked=True, ax=ax)
ax.set_title(f"Total produksi hortikultura per kecamatan ({tahun_geo})")
ax.set_xlabel("Produksi (kuintal)"); ax.set_ylabel("Kecamatan"); ax.legend(title="Kelompok")
plt.show()
t = prod["Total"]
interpretasi(f"Produksi tertinggi: {t.idxmax()} ({t.max():,.0f} kuintal, {t.max() / t.sum():.1%} dari total); "
             f"terendah: {t.idxmin()} ({t.min():,.0f}). Tiga kecamatan teratas menyumbang {t.nlargest(3).sum() / t.sum():.1%} dari produksi.")

# 6.2 Luas panen per kecamatan
luas = G.groupby("kecamatan")["luas_panen"].sum().sort_values()
fig, ax = plt.subplots(figsize=(9, 8))
luas.plot.barh(ax=ax, color="seagreen")
ax.set_title(f"Total luas panen hortikultura per kecamatan ({tahun_geo})")
ax.set_xlabel("Luas panen (ha)"); ax.set_ylabel("Kecamatan")
plt.show()
interpretasi(f"Luas panen terbesar: {luas.idxmax()} ({luas.max():,.0f} ha); terkecil: {luas.idxmin()} ({luas.min():,.0f} ha). "
             f"Rasio terbesar/terkecil = {luas.max() / max(luas.min(), 1e-9):,.1f}×.")

# 6.3 Wilayah dengan nilai tertinggi dan terendah
tabel = pd.DataFrame({"produksi_kuintal": t, "luas_panen_ha": luas}).sort_values("produksi_kuintal", ascending=False)
tabel["peringkat_produksi"] = np.arange(1, len(tabel) + 1)
tabel["peringkat_luas"] = tabel["luas_panen_ha"].rank(ascending=False).astype(int)
print("5 kecamatan tertinggi:"); display(tabel.head(5))
print("5 kecamatan terendah:"); display(tabel.tail(5))
selisih_rank = (tabel["peringkat_produksi"] - tabel["peringkat_luas"]).abs().idxmax()
interpretasi(f"Kecamatan dengan selisih peringkat produksi dan luas terbesar adalah {selisih_rank}: luasnya tidak sebanding "
             "dengan produksinya, yang dapat berarti komoditas bernilai produksi tinggi per hektar (mis. buah) atau data yang perlu dicek.")

# 6.4 Konsentrasi wilayah per komoditas (porsi kecamatan terhadap total komoditas, %)
komoditas_top = G.groupby("komoditas")["produksi_kuintal"].sum().nlargest(10).index
pk = G[G["komoditas"].isin(komoditas_top)].pivot_table(index="kecamatan", columns="komoditas",
                                                        values="produksi_kuintal", aggfunc="sum", fill_value=0)
porsi = (pk / pk.sum() * 100)[komoditas_top]
fig, ax = plt.subplots(figsize=(11, 9))
heatmap(ax, porsi, f"Porsi tiap kecamatan terhadap produksi kabupaten, 10 komoditas terbesar ({tahun_geo})",
        "Porsi produksi (%)", cmap="YlOrRd", vmin=0, vmax=100, anotasi_min=10, fmt="{:.0f}")
ax.set_xlabel("Komoditas"); ax.set_ylabel("Kecamatan")
plt.tight_layout()
plt.show()
sentra = [f"{k}: {porsi[k].idxmax()} ({porsi[k].max():.0f}%)" for k in komoditas_top[:5]]
interpretasi("Kecamatan penyumbang terbesar untuk komoditas teratas: " + "; ".join(sentra) +
             ". Nilai mendekati 100% berarti produksi terkonsentrasi di satu kecamatan (sentra).")

In [ ]:
# 7.1 Deteksi nilai ekstrem produktivitas PER KOMODITAS (robust z-score: median & MAD, pada skala log)
A = DF_AKTIF.copy()
A["log_prd"] = np.log10(A["produktivitas"])
grp = A.groupby("komoditas")["log_prd"]
A["n_obs"] = grp.transform("count")
A["z"] = grp.transform(robust_z)
q1, q3 = grp.transform(lambda s: s.quantile(0.25)), grp.transform(lambda s: s.quantile(0.75))
A["luar_iqr"] = (A["log_prd"] < q1 - 1.5 * (q3 - q1)) | (A["log_prd"] > q3 + 1.5 * (q3 - q1))
A["median_komoditas"] = grp.transform(lambda s: 10 ** s.median())
A["rasio_median"] = A["produktivitas"] / A["median_komoditas"]
A["porsi_kec"] = (DF["produksi_kuintal"] / DF.groupby(["tahun", "komoditas"])["produksi_kuintal"].transform("sum")).loc[A.index]

MIN_OBS = 8     # komoditas dengan pengamatan lebih sedikit tidak dinilai (statistik tidak stabil)
dinilai = A[A["n_obs"] >= MIN_OBS]
anomali = dinilai[dinilai["z"].abs() > Z_AMBANG].copy()
print(f"Komoditas dinilai: {dinilai['komoditas'].nunique()} (≥{MIN_OBS} pengamatan) | baris dinilai: {len(dinilai):,}")
print(f"Anomali produktivitas (|z| > {Z_AMBANG}): {len(anomali)} baris ({len(anomali) / len(dinilai):.1%}); "
      f"di luar batas IQR: {int(dinilai['luar_iqr'].sum())} baris")

# 7.2 Visualisasi: luas panen vs produktivitas, anomali ditandai
fig, ax = plt.subplots(figsize=(8, 5.5))
normal = dinilai.drop(anomali.index)
ax.scatter(normal["luas_panen"], normal["produktivitas"], s=8, alpha=0.3, color="gray", label=f"Normal (n={len(normal):,})")
ax.scatter(anomali["luas_panen"], anomali["produktivitas"], s=25, color="red", label=f"Anomali (n={len(anomali):,})")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_title("Deteksi anomali produktivitas (robust z-score per komoditas)")
ax.set_xlabel("Luas panen (ha, skala log)"); ax.set_ylabel("Produktivitas (kuintal/ha, skala log)")
ax.legend()
plt.show()
if len(anomali):
    kecil = (anomali["luas_panen"] < LUAS_KECIL_HA).mean()
    interpretasi(f"{len(anomali)} baris menyimpang jauh dari produktivitas tipikal komoditasnya. "
                 f"{kecil:.0%} di antaranya berluas panen < {LUAS_KECIL_HA} ha; nilai produktivitas pada luas sekecil itu sangat sensitif terhadap pembulatan.")
else:
    interpretasi("Tidak ada baris yang melewati ambang anomali; sebaran produktivitas per komoditas relatif rapat.")

# 7.3 Nilai ekstrem terbesar + kemungkinan penyebab (aturan berbasis karakteristik data)
def kemungkinan_penyebab(r):
    s = []
    if r["is_dummy"]:
        s.append("tahun data simulasi (variasi acak)")
    if r["luas_panen"] < LUAS_KECIL_HA:
        s.append("luas panen sangat kecil, produktivitas sensitif terhadap pembulatan")
    if r["porsi_kec"] > 0.5:
        s.append("kecamatan sentra komoditas (>50% produksi kabupaten)")
    if r["rasio_median"] > 3:
        s.append("jauh di atas median komoditas: cek satuan/metode pencatatan, atau budidaya sangat intensif")
    elif r["rasio_median"] < 1 / 3:
        s.append("jauh di bawah median: kemungkinan panen parsial/gagal panen atau pencatatan tidak lengkap")
    return "; ".join(s) or "tidak teridentifikasi dari data yang ada, perlu verifikasi ke sumber"

anomali["kemungkinan_penyebab"] = anomali.apply(kemungkinan_penyebab, axis=1)
kolom = ["tahun", "kecamatan", "komoditas", "luas_panen", "produksi_kuintal", "produktivitas",
         "median_komoditas", "rasio_median", "z", "kemungkinan_penyebab"]
print("15 anomali dengan |z| terbesar:")
with pd.option_context("display.max_colwidth", 90):
    display(anomali.reindex(anomali["z"].abs().sort_values(ascending=False).index).head(15)[kolom])
interpretasi("Kolom 'kemungkinan_penyebab' adalah hipotesis awal berdasarkan aturan sederhana pada data (luas kecil, konsentrasi "
             "wilayah, rasio terhadap median, status simulasi), bukan kesimpulan. Verifikasi ke sumber BPS/DKPP tetap diperlukan.")

# 7.4 Lonjakan produksi antar tahun (per kecamatan-komoditas)
if DF["tahun"].nunique() >= 2:
    P = DF.pivot_table(index=["kecamatan", "komoditas"], columns="tahun", values="produksi_kuintal", aggfunc="sum")
    th = sorted(P.columns)
    baris = []
    for t0, t1 in zip(th[:-1], th[1:]):
        ada = P[t0] > 0
        perubahan = (P.loc[ada, t1] - P.loc[ada, t0]) / P.loc[ada, t0]
        baris.append((t1, ada.sum(), (perubahan.abs() > LONJAKAN_YOY).mean() * 100))
    lonjak = pd.DataFrame(baris, columns=["tahun", "pasangan_dibandingkan", "persen_lonjakan"]).set_index("tahun")
    fig, ax = plt.subplots()
    ax.bar(label_tahun(lonjak.index), lonjak["persen_lonjakan"], color="indianred")
    ax.set_title(f"Porsi kecamatan-komoditas dengan perubahan produksi > {LONJAKAN_YOY:.0%} dari tahun sebelumnya")
    ax.set_xlabel("Tahun (* = data simulasi)"); ax.set_ylabel("Porsi pasangan yang melonjak (%)")
    plt.show()
    puncak = lonjak["persen_lonjakan"].idxmax()
    teks = (f"Lonjakan paling banyak terjadi pada {puncak} ({lonjak.loc[puncak, 'persen_lonjakan']:.0f}% pasangan). ")
    if TAHUN_DUMMY and puncak == min(t for t in th if t not in TAHUN_DUMMY):
        teks += ("Tahun ini adalah tahun pertama setelah data simulasi, sehingga lonjakan kemungkinan merupakan artefak pergantian "
                 "data simulasi ke data asli, bukan perubahan nyata di lapangan.")
    interpretasi(teks)

# 7.5 Ringkasan flag kualitas dari tahap perhitungan produktivitas
print("Distribusi flag kualitas (dari tahap produktivitas):")
display(DF["flag_kualitas"].value_counts().to_frame("jumlah_baris"))

In [ ]:
tahun_terakhir = int(DF["tahun"].max())
G = DF[DF["tahun"] == tahun_terakhir]
t = G.groupby("kecamatan")["produksi_kuintal"].sum()
top_kom = G.groupby("komoditas")["produksi_kuintal"].sum().nlargest(3)

print("RINGKASAN EDA HORTIKULTURA KABUPATEN KEDIRI")
print("-" * 60)
print(f"1. Data: {len(DF):,} baris, {DF['tahun'].nunique()} tahun ({DF['tahun'].min()}-{DF['tahun'].max()}), "
      f"{DF['kecamatan'].nunique()} kecamatan, {DF['komoditas'].nunique()} komoditas.")
print(f"2. Hanya {len(DF_AKTIF) / len(DF):.0%} kombinasi kecamatan-komoditas-tahun yang berpanen; sebaran sangat miring ke kanan.")
print(f"3. {tahun_terakhir}: kecamatan produksi terbesar {t.idxmax()} ({t.max() / t.sum():.0%}); "
      f"komoditas terbesar {', '.join(top_kom.index)}.")
print(f"4. Anomali produktivitas: {len(anomali)} baris (|z| > {Z_AMBANG}); lihat tabel kemungkinan penyebab di bagian 5.")
if TAHUN_DUMMY:
    print(f"5. ⚠️ Tahun {TAHUN_DUMMY} adalah data simulasi: tren temporal hanya ilustrasi metode, bukan temuan lapangan.")